# Analysis

## Setting parameters

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from quickrules.data_handling import calculate_score, count_all_rules, count_all_attributes, bold
from pathlib import Path
from quickrules.data_handling import balanced_accuracy_score
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd
import numpy as np

In [4]:
data_sets = [  # actual set for QR
    'australian',
     # 'automobile', # cat feats
     'bands',
     'bupa',
     'cleveland',
     # 'contraceptive', # no rules found
     # 'crx', # cat feats
     'dermatology',
     'ecoli',
     # 'german', # cat feats
     'glass',
     # 'haberman', # no rules found
     'heart',
     'ionosphere',
     # 'mammographic',  # no rules found
     'pima',
     # 'saheart', # cat feats
     'sonar',  # 70 features!
     'spectfheart',
     'vehicle',
     'vowel',
     'wine',
     'winequality-red',
     'wisconsin',
     'yeast'
]

def_sets = [
    'australian',
    'bupa',
    'cleveland',
    # 'crx',
    # 'dermatology',
    'ecoli',
    'glass',
    # 'haberman',
    'heart',
    'ionosphere',
    'pima',
    # 'sonar',
    'spectfheart',
    # 'saheart',
    # 'wdbc',
    'wine',
    'winequality-red',
    'wisconsin',
    'yeast',
    # 'automobile',
    # 'dermatology', run base again
    # 'german',
    # 'movement',
    # 'vehicle',
    # 'vowel',
]
control = [
    'australian',
    'bupa',
    'cleveland',
    # 'crx',
    'ecoli',
    'glass',
    'heart',
    'ionosphere',
    'pima',
    'spectfheart',
    'winequality-red',
    'wine',
    'wisconsin',
    'yeast'
]

In [5]:
# data_sets = control

In [6]:
len(data_sets)

18

In [7]:
data_base = Path.cwd() / 'keel-data'
include_sets = data_sets
metric = balanced_accuracy_score  # balanced_
scores = {}
rules = {}
attributes = {}
results_base = Path.cwd()

## Loading MODLEM results

In [8]:
weka_folder = Path.cwd() / 'weka_rules_files'
name = 'modlem'
file = 'weka-balanced-accuracy.csv'  # 'weka-accuracy.csv'

In [9]:
scores[name] = pd.read_csv(weka_folder / file, header=None, index_col=0).to_dict()[1]

In [10]:
rules[name] = {}
attributes[name] = {}
for file in weka_folder.iterdir():
    if file.name[-3:] == 'txt':
        short_name = file.name[:-4]
        with open(file, 'r') as f:
            line = f.readline()
            nrs = []
            while len(line) > 4:
                nrs.append(line.count('&') + 1)
                line = f.readline()
        rules[name][short_name] = len(nrs)
        attributes[name][short_name] = np.average(nrs)

## Loading QuickRules results

In [13]:
qr_models = {
    'qr': results_base / 'no-prune-results' / 'close-min-max-combo-results',
    # 'lin-owa': results_base / 'no-prune-results' / 'rules-lin-owa-qr-combo-minmax-results',
    # 'trun-lin-owa': results_base / 'no-prune-results' / 'rules-trun-lin-owa-qr-combo-minmax-results',
    # 'trun-exp-owa': results_base / 'no-prune-results' / 'trun-exp-owa-qr-combo-minmax-results',
    # 'avg-qr': results_base / 'mon-avg-std-minmax-results-2',
    # 'avg-lin-owa' : results_base / 'mon-avg-lin-owa-minmax-results-2',
    # 'prune-qr': results_base / 'prune-results' / 'prune-qr-combo-minmax-results',
    # 'prune-lin-owa': results_base / 'prune-results' / 'prune-lin-owa-qr-combo-minmax-results',
    # 'prune-trun-lin-owa': results_base / 'prune-results' / 'prune-trun-lin-owa-qr-combo-min-max-results',
    # 'prune-trun-exp-owa': results_base / 'prune-results' / 'prune-trun-exp-owa-qr-combo-min-max-results',
    # 'prune-avg-qr': results_base / 'prune-mon-avg-std-minmax-results-2',
    # 'prune-avg-lin-owa' : results_base / 'prune-mon-avg-lin-owa-minmax-results-2',
}

In [14]:
for model, path in qr_models.items():
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=include_sets,
        metric=metric
    )
    rules[model] = count_all_rules(
        Path.cwd() / results_base / path,
        include=include_sets
    )
    attributes[model] = count_all_attributes(
        Path.cwd() / results_base / path,
        include=include_sets
    )

## Adding results for non rule based models


In [15]:
other_models =  {
    'naive-bayes': Path.cwd() / 'NaiveBayes-results',
    'svm': Path.cwd() / 'svm',
    'tree': Path.cwd() / 'decision-tree'
}

In [16]:
for model, path in other_models.items():
    print(model)
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=include_sets,
        metric=metric,
        verbose=False
    )

naive-bayes
svm
tree


In [17]:
frnn_results = pd.read_csv(Path.cwd() / 'frnn_new_results.csv', header=None, index_col=0).to_dict()[1]
scores['frnn'] =  {data_set: frnn_results[data_set] for data_set in include_sets}

## NORI Models

In [18]:
nori_models = {
    # 'non-overlap': Path.cwd() / 'non-overlap-rules',
    # 'nori' : Path.cwd() / 'non-overlap-rules-def_sets',
    # 'lower-nori' : Path.cwd() / 'lower-approx-rules-def_sets',
    # 'lower-check' : Path.cwd() / 'lower-approx-og-implement-test',
    'lower-new-check' : Path.cwd() / 'lower-approx-new-impl-test'
}

In [19]:
for model, path in nori_models.items():
    scores[model] = calculate_score(
        data_folder=Path.cwd() / 'keel-data',
        results_folder=path, #'rule_prune-rel-0dot00-std',  #  'rel-0dot05-std', #' no-prune-results'/ 'close-min-max-combo-results',  # /
        metric=metric,
        include=data_sets,
        label_encoding=True
    )
    rules[model] = count_all_rules(
        path,
        include=data_sets
    )
    attributes[model] = count_all_attributes(
        path,
        include=include_sets,
        counter='{'
    )

In [20]:
scores.keys()

dict_keys(['modlem', 'qr', 'lin-owa', 'trun-lin-owa', 'trun-exp-owa', 'avg-qr', 'avg-lin-owa', 'prune-qr', 'prune-lin-owa', 'prune-trun-lin-owa', 'prune-trun-exp-owa', 'naive-bayes', 'svm', 'tree', 'frnn', 'lower-new-check'])

## Tables
set 1 = QR + OWA-variants without pruning
set 2 = focus on pruning

In [21]:
 names = {
    1: [
        'qr',
        'lin-owa',
        'trun-lin-owa',
        'trun-exp-owa',
        'modlem'
    ],
    2: [
        'qr',
        'lin-owa',
        'prune-qr',
        'prune-lin-owa'
    ],
    3: [
        'qr',
        'lin-owa',
        'avg-qr',
        'avg-lin-owa'
    ],
    4: [
        'lin-owa',
        'modlem'
    ],
    5: [
        'lin-owa',
        'frnn',
        'svm',
        'naive-bayes',
        'tree',
    ],
    6: [
        # 'nori',
        # 'lower-nori',
        # 'lower-check',
        'lower-new-check',
        'modlem',
        'qr',
    ]
}

In [22]:
nr = 6

In [23]:
main_folder = 'balanced_acc'  # 'normal_acc'
tables_path_base = Path.cwd() / 'tables' / main_folder

In [24]:
table_acc = pd.DataFrame(index=data_sets, columns=names[nr])

In [25]:
for model in names[nr]:
    for data_set in data_sets:
    # for data_set, score in scores[model].items():
        table_acc.loc[data_set, model] = scores[model][data_set]

In [26]:
table_acc.loc['mean'] = table_acc.mean(axis='rows', skipna=True)

In [27]:
table_acc

,lower-new-check,modlem,qr
australian,0.787517,0.861,0.865348
bands,0.675141,0.6035,0.533045
bupa,0.600724,0.6525,0.5
cleveland,0.290397,0.2764,0.208347
dermatology,0.895363,0.941333,0.521995
ecoli,0.720659,0.512,0.17585
glass,0.679067,0.518333,0.31381
heart,0.7675,0.7605,0.785833
ionosphere,0.912468,0.8905,0.658974
pima,0.687633,0.6985,0.502775


In [28]:
bolded_first = table_acc.apply(lambda data: bold(data), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_60233/3326315922.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)


In [29]:
table_rule = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, rule_count in rules[model].items():
    for data_set in data_sets:
        table_rule.loc[data_set, model] = rules[model][data_set]  #  rule_count
table_rule.loc['mean'] = table_rule.mean(axis='rows', skipna=True)

In [30]:
table_rule

,lower-new-check,modlem,qr
australian,92.3,121,732.2
bands,59.0,113,222.7
bupa,76.7,103,332.4
cleveland,102.2,95,331.1
dermatology,20.2,27,90.5
ecoli,56.8,56,315.0
glass,51.1,50,225.7
heart,44.7,62,303.7
ionosphere,19.3,30,460.5
pima,125.0,191,824.6


In [31]:
bolded_first = table_rule.apply(lambda data: bold(data, optimum='min', format_string="%.1f"), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_60233/1993066615.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)


In [32]:
table_attribute = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, attribute_count in attributes[model].items():
    for data_set in data_sets:
        table_attribute.loc[data_set, model] = attributes[model][data_set]  # attribute_count
table_attribute.loc['mean'] = table_attribute.mean(axis='rows', skipna=True)

In [33]:
table_attribute

,lower-new-check,modlem,qr
australian,15.952302,2.355372,8.04978
bands,19.321915,2.106195,1.753697
bupa,13.693114,2.194175,4.40226
cleveland,16.743198,2.589474,8.08194
dermatology,14.294033,2.555556,6.726801
ecoli,12.52774,2.339286,3.816832
glass,12.401252,2.1,5.343654
heart,14.973593,2.209677,7.580314
ionosphere,13.653764,1.533333,11.653025
pima,16.285493,2.073298,5.687857


In [34]:
bolded_first = table_attribute.apply(lambda data: bold(data, optimum='min'), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_60233/3339941083.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)


## Statistical testing

In [57]:
from scipy import stats
import scikit_posthocs as sph

In [99]:
subject = table_acc

In [100]:
no_mean =  subject.drop(labels = 'mean', axis = 'index')

In [101]:
no_mean

,lower-new-check,modlem,qr
australian,0.787517,0.861,0.865348
bands,0.675141,0.6035,0.533045
bupa,0.600724,0.6525,0.5
cleveland,0.290397,0.2764,0.208347
dermatology,0.895363,0.941333,0.521995
ecoli,0.720659,0.512,0.17585
glass,0.679067,0.518333,0.31381
heart,0.7675,0.7605,0.785833
ionosphere,0.912468,0.8905,0.658974
pima,0.687633,0.6985,0.502775


In [104]:
stats.wilcoxon(no_mean['lower-new-check'], no_mean['modlem'], alternative="greater")


WilcoxonResult(statistic=104.0, pvalue=0.22114944458007812)

In [95]:
f_test = stats.friedmanchisquare(*no_mean.values)
f_test

FriedmanchisquareResult(statistic=45.31578947368419, pvalue=0.000218491354338183)

In [102]:
post_hocs = sph.posthoc_nemenyi_friedman(no_mean.values) # , p_adjust='Holm'
post_hocs.columns=no_mean.columns
post_hocs.index=no_mean.columns
post_hocs

,lower-new-check,modlem,qr
lower-new-check,1.000,0.900000,0.001000
modlem,0.900,1.000000,0.002474
qr,0.001,0.002474,1.000000
